## Validation with DOE data

In [1]:
cd G:\My Drive\WVU_academic\EE-593A\project\input_files

G:\My Drive\WVU_academic\EE-593A\project\input_files


In [15]:
import pandas as pd

states = ['MA','ME','NH','NY','VT','CT']
# states = ['CT']
for st in states:
    path = f"G:/My Drive/WVU_academic/EE-593A/project/input_files/{st}_monthlytweets_with_outage.csv"
    df = pd.read_csv(path)
    df['Local Date'] = (pd.to_datetime(df['Local Timestamp'],format='%I:%M %p - %d %b %Y').dt.strftime('%m/%d/%Y'))
    df = df[['Local Date','State','Outage_status']]
    out_path = f"G:/My Drive/WVU_academic/EE-593A/project/input_files/{st}_monthlytweets_with_outage_updated.csv"
    df.to_csv(out_path, index=False)

In [8]:
df['Local Timestamp']

0       3:10 PM - 29 Dec 2010
1       1:07 PM - 29 Dec 2010
2       1:01 PM - 29 Dec 2010
3        9:02 AM - 7 Jan 2019
4       1:20 AM - 29 Dec 2010
                ...          
4583     7:44 PM - 4 Jan 2012
4584     1:25 PM - 3 Jan 2012
4585     9:34 PM - 2 Jan 2012
4586     8:25 AM - 2 Jan 2012
4587     1:08 PM - 1 Jan 2012
Name: Local Timestamp, Length: 4588, dtype: object

In [12]:
df['Local Date'] = (pd.to_datetime(df['Local Timestamp'],format='%I:%M %p - %d %b %Y').dt.strftime('%m/%d/%Y'))

In [11]:
df = df[['Screen Name','Status ID','Text','Local Timestamp','Local Date','Likes','Retweets','State','Outage_status']]

Index(['Screen Name', 'Status ID', 'Text', 'Local Timestamp', 'Likes',
       'Retweets', 'State', 'Outage_status', 'Local Date'],
      dtype='object')

In [17]:
import pandas as pd

states = ['MA', 'ME', 'NH', 'NY', 'VT', 'CT']
for st in states:
    path = f"G:/My Drive/WVU_academic/EE-593A/project/input_files/{st}_monthlytweets_with_outage.csv"
    df = pd.read_csv(path)

    df['Local Date'] = pd.to_datetime(
        df['Local Timestamp'],
        format='%I:%M %p - %d %b %Y'
    )

    start, end = '2009-09-01', '2012-12-31'
    df = df.loc[df['Local Date'].between(start, end)].copy()

    df['Local Date'] = df['Local Date'].dt.strftime('%m/%d/%Y')

    df = df[['Local Date', 'State', 'Outage_status']]

    out_path = f"G:/My Drive/WVU_academic/EE-593A/project/input_files/{st}_monthlytweets_with_outage_updated.csv"
    df.to_csv(out_path, index=False)


In [18]:
import pandas as pd
df_doe = pd.read_csv("Grid_Disruption_00_14_standardized_cleaned.csv")
df_doe.columns

Index(['Event Description', 'Year', 'Date Event Began', 'Time Event Began',
       'Date of Restoration', 'Time of Restoration', 'Respondent',
       'Geographic Areas', 'NERC Region', 'Demand Loss (MW)',
       'Number of Customers Affected', 'Tags'],
      dtype='object')

In [20]:
import pandas as pd
import re

df = df_doe.copy()

df['event_dt'] = pd.to_datetime(df['Date Event Began'].str.strip() + ' ' + df['Time Event Began'].fillna(''), errors='coerce')
df = df.loc[(df['event_dt'] >= '2009-09-01') & (df['event_dt'] <= '2012-12-31')]

state_map = {
    'Massachusetts': 'MA',
    'Maine': 'ME',
    'New Hampshire': 'NH',
    'New York': 'NY',
    'Vermont': 'VT',
    'Connecticut': 'CT'
}

def extract_state_code(text):
    for full, abbr in state_map.items():
        if pd.notna(text) and re.search(rf'\b{re.escape(full)}\b', text, re.IGNORECASE):
            return abbr
    return None

df['State'] = df['Geographic Areas'].apply(extract_state_code)
df = df[df['State'].notna()]
df['Event Date'] = df['event_dt'].dt.strftime('%m/%d/%Y')
df = df.sort_values(['event_dt', 'State'])
result = df[['Event Description','Year','Event Date','State']].copy()
result['Outage_status'] = 1
output_path = r"G:\My Drive\WVU_academic\EE-593A\project\input_files\doe_monthlytweets_with_outage_updated.csv"
result.to_csv(output_path, index=False)

In [22]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import imageio
from io import BytesIO

# 1) Prepare your data (same as before)
df2 = result.copy()
df2['Event Date'] = pd.to_datetime(df2['Event Date'], format='%m/%d/%Y')
df_count = (
    df2
    .groupby(['Event Date', 'State'], as_index=False)
    .size()
    .rename(columns={'size':'Outages'})
)
df_count['DateStr'] = df_count['Event Date'].dt.strftime('%Y-%m-%d')
frames = []

# 2) Render each date as a static frame
pio.kaleido.scope.default_format = "png"
for date in sorted(df_count['DateStr'].unique()):
    sub = df_count[df_count['DateStr']==date]
    fig = px.choropleth(
        sub,
        locations='State',
        locationmode='USA-states',
        color='Outages',
        scope='usa',
        color_continuous_scale='Reds',
        title=f"Outages on {date}"
    )
    fig.update_layout(geo=dict(bgcolor='rgba(0,0,0,0)'),
                      margin=dict(l=0,r=0,t=40,b=0))
    # export to a PNG in memory
    img_bytes = fig.to_image(width=800, height=500)
    frames.append(imageio.imread(BytesIO(img_bytes)))

# 3) Stitch into a GIF
imageio.mimsave('us_outages.gif', frames, fps=1)

C:\Users\mr00065\AppData\Local\Temp\ipykernel_40116\633656038.py:36: DeprecationWarning:

Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning dissapear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.

C:\Users\mr00065\AppData\Local\Temp\ipykernel_40116\633656038.py:36: DeprecationWarning:

Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning dissapear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.

C:\Users\mr00065\AppData\Local\Temp\ipykernel_40116\633656038.py:36: DeprecationWarning:

Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning dissapear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.

C:\Users\mr00065\AppData\Local\Temp\ipyk

In [46]:
import pandas as pd

states = ['MA','ME','NH','NY','VT','CT']
all_df = pd.concat([pd.read_csv(f"G:/My Drive/WVU_academic/EE-593A/project/input_files/{st}_monthlytweets_with_outage.csv") for st in states], ignore_index=True)
all_df.to_csv("G:/My Drive/WVU_academic/EE-593A/project/input_files/all_monthlytweets_with_outage.csv", index=False)

In [48]:
all_df.tail()

,Screen Name,Status ID,Text,Local Timestamp,Likes,Retweets,State,Outage_status
19025,DiMatteo176,154724840740368384.0,New legend power 2012 #PINKSTUDS @ J&Jracing ...,7:44 PM - 4 Jan 2012,Like\n \n \n \n \n \n \n ...,Retweet\n \n \n \n \n \n \n ...,CT,1
19026,FamousAssRyann,154267266433822720.0,"Money, Power, Nd Fame Right Now'",1:25 PM - 3 Jan 2012,Like\n \n \n \n \n \n \n ...,Retweet\n \n \n 1\n \n \n \n ...,CT,1
19027,aliciaanop,154027879791992832.0,@Anniekamradt RIP squirt. #bestmemory when my ...,9:34 PM - 2 Jan 2012,Like\n \n \n \n \n \n \n ...,Retweet\n \n \n \n \n \n \n ...,CT,0
19028,thesarahdipity,153829239194517504.0,@MikeMGarner omg ashley won so kool I knew she...,8:25 AM - 2 Jan 2012,Like\n \n \n \n \n \n \n ...,Retweet\n \n \n \n \n \n \n ...,CT,1
19029,maeberganbarber,153538034489495552.0,Absolutely beautiful day - power walk!,1:08 PM - 1 Jan 2012,Like\n \n \n \n \n \n \n ...,Retweet\n \n \n \n \n \n \n ...,CT,1


In [52]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import imageio
from io import BytesIO
import imageio.v2 as imageio

# 1) Prepare your data (same as before)
df2 = all_df.copy()

df2[['Time','DateStr0']] = df2['Local Timestamp'].str.split(' - ', expand=True)
df2['Event Date'] = pd.to_datetime(df2['DateStr0'], format='%d %b %Y')

df2['Event Date'] = pd.to_datetime(df2['Event Date'], format='%m/%d/%Y')
df_count = (
    df2
    .groupby(['Event Date', 'State'], as_index=False)
    .size()
    .rename(columns={'size':'Outages'})
)
df_count['DateStr'] = df_count['Event Date'].dt.strftime('%Y-%m-%d')
frames = []

# 2) Render each date as a static frame
pio.kaleido.scope.default_format = "png"
for date in sorted(df_count['DateStr'].unique()):
    sub = df_count[df_count['DateStr']==date]
    fig = px.choropleth(
        sub,
        locations='State',
        locationmode='USA-states',
        color='Outages',
        scope='usa',
        color_continuous_scale='Reds',
        title=f"Outages on {date}"
    )
    fig.update_layout(geo=dict(bgcolor='rgba(0,0,0,0)'),
                      margin=dict(l=0,r=0,t=40,b=0))
    # export to a PNG in memory
    img_bytes = fig.to_image(width=800, height=500)
    frames.append(imageio.imread(BytesIO(img_bytes)))

# 3) Stitch into a GIF
num_frames = len(frames)
total_time = 20.0                 # seconds
fps = num_frames / total_time 


imageio.mimsave('us_outages_all.gif', frames, fps=1)

C:\Users\mr00065\AppData\Local\Temp\ipykernel_40116\488869262.py:40: DeprecationWarning:

Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning dissapear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.

C:\Users\mr00065\AppData\Local\Temp\ipykernel_40116\488869262.py:40: DeprecationWarning:

Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning dissapear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.

C:\Users\mr00065\AppData\Local\Temp\ipykernel_40116\488869262.py:40: DeprecationWarning:

Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning dissapear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.

C:\Users\mr00065\AppData\Local\Temp\ipyk

In [54]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import imageio.v2 as imageio
from io import BytesIO
# 1) Prepare your data (same as before)
df2 = all_df.copy()

df2[['Time','DateStr0']] = df2['Local Timestamp'].str.split(' - ', expand=True)
df2['Event Date'] = pd.to_datetime(df2['DateStr0'], format='%d %b %Y')

df2['Event Date'] = pd.to_datetime(df2['Event Date'], format='%m/%d/%Y')
df_count = (
    df2
    .groupby(['Event Date', 'State'], as_index=False)
    .size()
    .rename(columns={'size':'Outages'})
)
df_count['DateStr'] = df_count['Event Date'].dt.strftime('%Y-%m-%d')
frames = []

pio.kaleido.scope.default_format = "png"
for date in sorted(df_count['DateStr'].unique()):
    sub = df_count[df_count['DateStr'] == date]
    fig = px.choropleth(
        sub,
        locations='State',
        locationmode='USA-states',
        color='Outages',
        scope='usa',
        color_continuous_scale='Reds',
        title=f"Outages on {date}"
    )
    fig.update_layout(geo=dict(bgcolor='rgba(0,0,0,0)'), margin=dict(l=0, r=0, t=40, b=0))
    img_bytes = fig.to_image(width=800, height=500)
    frames.append(imageio.imread(BytesIO(img_bytes)))

subsample_rate = max(1, len(frames) // 1000)
frames_small = frames[::subsample_rate]
duration = 20.0 / len(frames_small)
imageio.mimsave('us_outages_all.gif', frames_small, duration=duration)